# Session 03 Practical Activity – Data Cleaning Mini Task


## Objective

This notebook demonstrates the complete cleaning process for the messy workplace dataset and prepares the data for machine learning.


## 1. Import Libraries

In [15]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler


## 2. Load Dataset

In [16]:

df = pd.read_csv("session03datacleaningandpreprocessing20260509t102937z3001zip-d7vgpjcrtl0reb7jhg70/Session_03_Data_Cleaning_and_Preprocessing/uk_workplace_data_cleaning_messy_dataset.csv")

df.head()


,learner_id,department,region,role_family,employee_band,training_type,enrolment_date,prior_python_score,course_minutes,engagement_score,manager_support_score,support_tickets_last_30d,completion_days,completed_on_time
0,L10387,OPERATIONS,Remote,Technician,A,ML Foundations,15-02-2026,41.9,189.8,91.3,4.0,1,20.0,1
1,L10089,Sales,Unknown,Associate,A,ML Foundations,07-02-2026,39.6,191.8,84.7,NaN,1,23.0,1
2,L10448,operations,Manchester,Analyst,B,AI Awareness,10-03-2026,36.1,93.6,100.0,1.0,3,13.0,1
3,L10599,Human Resources,Manchester,Technician,C,AI Awareness,05-01-2026,55.7,186.0,37.1,4.0,4,8.0,0
4,L10227,IT,Edinburgh,Coordinator,B,Data Literacy,08-03-2026,10.6,237.5,47.9,1.0,1,15.0,0


## 3. Initial Inspection

In [17]:

print(df.shape)

print(df.isnull().sum())

print(df.duplicated().sum())

df.info()


(620, 14)
learner_id                   0
department                  18
region                      24
role_family                  0
employee_band                0
training_type                0
enrolment_date               0
prior_python_score          35
course_minutes               0
engagement_score            30
manager_support_score       14
support_tickets_last_30d     0
completion_days             18
completed_on_time            0
dtype: int64
20
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 620 entries, 0 to 619
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   learner_id                620 non-null    object 
 1   department                602 non-null    object 
 2   region                    596 non-null    object 
 3   role_family               620 non-null    object 
 4   employee_band             620 non-null    object 
 5   training_type             620 non-null    obj

## 4. Remove Duplicate Rows

In [18]:

df = df.drop_duplicates()

print("Duplicates removed")


Duplicates removed


## 5. Standardise Category Labels

In [19]:

df['department'] = df['department'].replace({
    ' OPERATIONS ': 'Operations',
    'operations': 'Operations',
    'finance': 'Finance',
    'sales': 'Sales',
    'Human Resources': 'HR',
    'Information Technology': 'IT'
})

df['region'] = df['region'].replace({
    'REMOTE': 'Remote',
    'remote ': 'Remote',
    'Unknown': np.nan
})


## 6. Handle Missing Values

In [20]:

df['department'] = df['department'].fillna(df['department'].mode()[0])

df['region'] = df['region'].fillna(df['region'].mode()[0])


## 7. Convert Support Tickets into Numeric Format

In [21]:

df['support_tickets_last_30d'] = (
    df['support_tickets_last_30d']
    .astype(str)
    .str.extract(r'(\d+)')
    .astype(int)
)


## 8. Check Impossible Numeric Values

In [22]:

df.loc[
    (df['prior_python_score'] < 0) |
    (df['prior_python_score'] > 100),
    'prior_python_score'
] = np.nan

df.loc[
    df['course_minutes'] < 0,
    'course_minutes'
] = np.nan

df.loc[
    df['engagement_score'] > 100,
    'engagement_score'
] = np.nan


## 9. Fill Missing Numerical Values

In [23]:

numeric_cols = [
    'prior_python_score',
    'course_minutes',
    'engagement_score',
    'manager_support_score',
    'completion_days'
]

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())


## 10. Feature Engineering

In [24]:

le = LabelEncoder()

df['department_encoded'] = le.fit_transform(df['department'])

scaler = StandardScaler()

df['prior_python_score_scaled'] = scaler.fit_transform(
    df[['prior_python_score']]
)

df.head()


,learner_id,department,region,role_family,employee_band,training_type,enrolment_date,prior_python_score,course_minutes,engagement_score,manager_support_score,support_tickets_last_30d,completion_days,completed_on_time,department_encoded,prior_python_score_scaled
0,L10387,Operations,Remote,Technician,A,ML Foundations,15-02-2026,41.9,189.8,91.3,4.0,1,20.0,1,4,-0.652858
1,L10089,Sales,Remote,Associate,A,ML Foundations,07-02-2026,39.6,191.8,84.7,3.0,1,23.0,1,5,-0.788043
2,L10448,Operations,Manchester,Analyst,B,AI Awareness,10-03-2026,36.1,93.6,100.0,1.0,3,13.0,1,4,-0.993760
3,L10599,HR,Manchester,Technician,C,AI Awareness,05-01-2026,55.7,186.0,37.1,4.0,4,8.0,0,2,0.158255
4,L10227,IT,Edinburgh,Coordinator,B,Data Literacy,08-03-2026,10.6,237.5,47.9,1.0,1,15.0,0,3,-2.492556


## 11. Before and After Diagnostics

In [25]:

diagnostics = pd.DataFrame({
    'Metric': ['Rows', 'Columns'],
    'Before': [102, 12],
    'After': [df.shape[0], df.shape[1]]
})

diagnostics


,Metric,Before,After
0,Rows,102,600
1,Columns,12,16


## 12. Save Cleaned Dataset

In [26]:

df.to_csv('cleaned_uk_workplace_dataset.csv', index=False)

print("Dataset saved successfully")


Dataset saved successfully



## 13. Bias, Signal Preservation, and Leakage

### Bias
Rows were not removed unnecessarily to avoid introducing bias.

### Signal Preservation
Outliers were reviewed carefully instead of automatically deleting them.

### Leakage
The target variable was not modified carelessly during preprocessing.
